# FISBe EDA


Download process

```bash
nohup bash -c '
  mkdir -p fisbe &&
  echo "[$(date)] Starting download..." &&
  curl -L -s -w "[$(date)] Download complete: %{size_download} bytes, %{time_total}s\n" \
    https://zenodo.org/api/records/10875063/files-archive -o fisbe/archive.zip &&
  echo "[$(date)] Extracting..." &&
  unzip -o fisbe/archive.zip -d fisbe/ &&
  rm fisbe/archive.zip &&
  echo "[$(date)] Done."
' > download.txt 2>&1 &

echo "PID: $!"
```

In [1]:
import os
from pathlib import Path

# If the kernel starts in .../AlmaAI/ipynb
root = Path.cwd()
if root.name == "ipynb":
    root = root.parent
os.chdir(root)


## Open File

In [2]:
import json
import napari
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

DATASET = Path('./fisbe/completely').resolve()
train_path = DATASET / 'train'
test_path = DATASET / 'test'
val_path = DATASET / 'val'
file_name = "R38F04-20181005_63_G3.zarr"



/home/william-zheng/.local/share/mamba/envs/pyimagej/lib/python3.9/site-packages/numpydoc/docscrape.py:203: UserWarning: potentially wrong underline length... 
Extended Summary 
---------- in 
Add a Points layer to the layer list. 
...
  while not self._is_at_section() and not self._doc.eof():
/home/william-zheng/.local/share/mamba/envs/pyimagej/lib/python3.9/site-packages/numpydoc/docscrape.py:203: UserWarning: potentially wrong underline length... 
Extended Summary 
---------- in 
Add a Labels layer to the layer list. 
...
  while not self._is_at_section() and not self._doc.eof():
/home/william-zheng/.local/share/mamba/envs/pyimagej/lib/python3.9/site-packages/numpydoc/docscrape.py:203: UserWarning: potentially wrong underline length... 
Extended Summary 
---------- in 
Add a Shapes layer to the layer list. 
...
  while not self._is_at_section() and not self._doc.eof():
/home/william-zheng/.local/share/mamba/envs/pyimagej/lib/python3.9/site-packages/numpydoc/docscrape.py:203: UserWar

## Create tiff Version for BiaPy

In [8]:
import zarr
import shutil
from tifffile import imwrite
from tqdm import tqdm


# Create x_y_dir for BiaPy
def create_x_y_dir(path:Path):
    (path / 'raw').mkdir(parents=True, exist_ok=True)
    (path / 'label').mkdir(parents=True, exist_ok=True)

def create_dirs(root_dir:Path, dirs: list[Path]):
    for dir in dirs:
        (root_dir / dir).mkdir(parents=True, exist_ok=True)
        create_x_y_dir(root_dir / dir)

TIFF_PATH = Path('./fisbe/biapy').resolve()
shutil.rmtree(TIFF_PATH, ignore_errors=True)
TIFF_PATH.mkdir(parents=True, exist_ok=True)
create_dirs(TIFF_PATH, ['train', 'test', 'val'])
print(DATASET)
print(TIFF_PATH)

/home/william-zheng/Documents/Programming/Python/NeuroInformatics/summer_2026/neuroinfo_fruitfly/fisbe/completely
/home/william-zheng/Documents/Programming/Python/NeuroInformatics/summer_2026/neuroinfo_fruitfly/fisbe/biapy


In [10]:
raw = zarr.open(DATASET /  'train' / 'R38F04-20181005_63_G3.zarr', mode='r', path="volumes/raw")
seg = zarr.open(DATASET /  'train' / 'R38F04-20181005_63_G3.zarr', mode='r', path="volumes/gt_instances")
raw_np = np.array(raw)
seg_np = np.array(seg)
raw_np.shape, seg_np.shape, np.unique(seg_np)

((3, 431, 687, 684), (2, 431, 687, 684), array([0, 1, 2], dtype=uint8))

In [9]:
train_files = os.listdir(train_path)
test_files = os.listdir(test_path)
val_files = os.listdir(val_path)

def create_tiff_version(data_path, tiff_path=TIFF_PATH):
    for file in tqdm(os.listdir(data_path)):
        raw = zarr.open(data_path / file, mode='r', path="volumes/raw")
        seg = zarr.open(data_path / file, mode='r', path="volumes/gt_instances")
        raw_np = np.array(raw)
        seg_np = np.array(seg)  
        raw_np.shape, seg_np.shape, np.unique(seg_np)
        imwrite(tiff_path / 'raw' / f"{file}.tiff", raw_np)
        imwrite(tiff_path / 'label' / f"{file}_seg.tiff", seg_np)

# Create tiff version for BiaPy
for split in ['train', 'test', 'val']:
    tqdm.write(f"Running Split {split}")
    create_tiff_version(DATASET / split, TIFF_PATH / split)


Running Split train


100%|██████████| 18/18 [03:17<00:00, 10.99s/it]


Running Split test


100%|██████████| 7/7 [00:58<00:00,  8.32s/it]


Running Split val


100%|██████████| 5/5 [00:33<00:00,  6.70s/it]


In [ ]:
!conda activate BiaPy_env
!python fisbe/biapy/prepare_tiff_data.py --splits test --clean
# add train val to --splits when you have those zarr splits ready

In [ ]:
# viewer = napari.Viewer(ndisplay=3)
# for idx, gt in enumerate(seg):
#   viewer.add_labels(
#     gt, rendering='translucent', blending='additive', name=f'gt_{idx}')
# viewer.add_image(raw[0], colormap="red", name='raw_r', blending='additive')
# viewer.add_image(raw[1], colormap="green",  name='raw_g', blending='additive')
# viewer.add_image(raw[2], colormap="blue",  name='raw_b', blending='additive')
# napari.run()